# LLM

In [1]:
import argparse
from xuance.common import get_configs
from xuance.environment import make_envs
from xuance.environment import REGISTRY_ENV
from xuance.environment import make_envs
from xuance.torch.agents import DDPG_Agent, PPOCLIP_Agent, DreamerV3Agent
from agents.DiffPPO import DiffusionPPOAgent  # class custom bạn đã viết


In [12]:
from xuance.common import get_configs
from xuance.environment import REGISTRY_ENV, make_envs
import numpy as np
import importlib
import env.LLM_env as llm_env_module

configs_dict = get_configs(file_dir="configs/DiffPPO_config_LLM.yaml")
configs = argparse.Namespace(**configs_dict)

# Register custom LLM environment.
importlib.reload(llm_env_module)
REGISTRY_ENV[configs.env_name] = llm_env_module.UAVLLMOffloadingEnv

# Quick smoke check for vectorized env construction and reset.
envs = make_envs(configs)
obs, info = envs.reset()
print('envs ready:', type(envs).__name__)
print('obs shape:', obs.shape if isinstance(obs, np.ndarray) else type(obs))
print('info len:', len(info) if isinstance(info, list) else type(info))
envs.close()

# Train DiffPPO agent.
envs = make_envs(configs)
Agent = DiffusionPPOAgent(config=configs, envs=envs)
Agent.train(configs.running_steps // configs.parallels)
Agent.save_model("final_train_model.pth")
Agent.finish()
envs.close()

envs ready: DummyVecEnv
obs shape: (30, 43)
info len: 30


  0%|          | 0/5000 [00:00<?, ?it/s]

100%|██████████| 5000/5000 [01:45<00:00, 47.42it/s]


In [13]:

# Load config (reuse DDPG-style training settings).
configs_dict = get_configs(file_dir="configs/PPO_LLM.yaml")
configs = argparse.Namespace(**configs_dict)

# Register custom LLM environment.
importlib.reload(llm_env_module)
REGISTRY_ENV[configs.env_name] = llm_env_module.UAVLLMOffloadingEnv

# Quick smoke check for vectorized env construction and reset.
envs = make_envs(configs)
obs, info = envs.reset()
print('envs ready:', type(envs).__name__)
print('obs shape:', obs.shape if isinstance(obs, np.ndarray) else type(obs))
print('info len:', len(info) if isinstance(info, list) else type(info))
envs.close()

# Train PPO agent.
envs = make_envs(configs)
Agent = PPOCLIP_Agent(config=configs, envs=envs)
Agent.train(configs.running_steps // configs.parallels)
Agent.save_model("final_train_model.pth")
Agent.finish()
envs.close()

envs ready: DummyVecEnv
obs shape: (30, 43)
info len: 30


  0%|          | 0/5000 [00:00<?, ?it/s]

100%|██████████| 5000/5000 [00:26<00:00, 190.27it/s]


In [14]:

# Load config (reuse DDPG-style training settings).
configs_dict = get_configs(file_dir="configs/DDPG_LLM_env.yaml")
configs = argparse.Namespace(**configs_dict)

# Register custom LLM environment.
importlib.reload(llm_env_module)
REGISTRY_ENV[configs.env_name] = llm_env_module.UAVLLMOffloadingEnv

# Quick smoke check for vectorized env construction and reset.
envs = make_envs(configs)
obs, info = envs.reset()
print('envs ready:', type(envs).__name__)
print('obs shape:', obs.shape if isinstance(obs, np.ndarray) else type(obs))
print('info len:', len(info) if isinstance(info, list) else type(info))
envs.close()
# Train DDPG agent.
envs = make_envs(configs)
Agent = DDPG_Agent(config=configs, envs=envs)
Agent.train(configs.running_steps // configs.parallels)
Agent.save_model("final_train_model.pth")
Agent.finish()
envs.close()

envs ready: DummyVecEnv
obs shape: (30, 43)
info len: 30


  0%|          | 0/5000 [00:00<?, ?it/s]

100%|██████████| 5000/5000 [00:28<00:00, 176.38it/s]


In [15]:
if not hasattr(DreamerV3Agent, "envs"):
    DreamerV3Agent.envs = property(lambda self: self.train_envs)
    
# Load config (reuse DDPG-style training settings).
configs_dict = get_configs(file_dir="configs/Dreamer_LLM_env.yaml")
configs = argparse.Namespace(**configs_dict)

# Register custom LLM environment.
importlib.reload(llm_env_module)
REGISTRY_ENV[configs.env_name] = llm_env_module.UAVLLMOffloadingEnv


# quick smoke run first
Agent = DreamerV3Agent(config=configs, envs=envs)
Agent.train(configs.running_steps // configs.parallels)
Agent.save_model("dreamerv3_smoke_model.pth")
Agent.finish()

  0%|          | 2/5000 [00:00<00:46, 107.91it/s]


AssertionError: 

In [11]:
# Reward diagnostics for UAVLLMOffloadingEnv
import argparse
import importlib
import numpy as np
from xuance.common import get_configs
from xuance.environment import REGISTRY_ENV, make_envs
import env.LLM_env as llm_env_module

configs_dict = get_configs(file_dir="configs/DDPG_LLM_env.yaml")
configs = argparse.Namespace(**configs_dict)
importlib.reload(llm_env_module)
REGISTRY_ENV[configs.env_name] = llm_env_module.UAVLLMOffloadingEnv

envs = make_envs(configs)
obs, info = envs.reset()

reward_list = []
u_term, l_term, p_term, pen_term = [], [], [], []

num_envs = int(configs.parallels)
action_dim = int(envs.action_space.shape[0])

for _ in range(30):
    action = np.random.uniform(-1.0, 1.0, size=(num_envs, action_dim)).astype(np.float32)
    obs, rewards, terminated, truncated, infos = envs.step(action)

    rewards_np = np.asarray(rewards, dtype=np.float32).reshape(-1)
    reward_list.extend(rewards_np.tolist())

    if isinstance(infos, (list, tuple)) and len(infos) > 0:
        for item in infos:
            u_term.append(float(item.get("reward_utility_term", 0.0)))
            l_term.append(float(item.get("reward_latency_term", 0.0)))
            p_term.append(float(item.get("reward_power_term", 0.0)))
            pen_term.append(float(item.get("reward_penalty_term", 0.0)))

    if np.any(np.asarray(terminated)) or np.any(np.asarray(truncated)):
        break

envs.close()

print("reward mean/min/max:", float(np.mean(reward_list)), float(np.min(reward_list)), float(np.max(reward_list)))
print("utility term mean:", float(np.mean(u_term)) if len(u_term) else 0.0)
print("latency term mean:", float(np.mean(l_term)) if len(l_term) else 0.0)
print("power term mean:", float(np.mean(p_term)) if len(p_term) else 0.0)
print("penalty term mean:", float(np.mean(pen_term)) if len(pen_term) else 0.0)

reward mean/min/max: -9990.029231770834 -10000.0 -9700.876953125
utility term mean: 0.7691913333333333
latency term mean: 1209.5055256873923
power term mean: 2.380646178325017
penalty term mean: -12065.381145511707
